# RVI-Kenya — Notebook 03: Flood Hub validation (RQ2 – RQ4)

This notebook addresses three of the proposal's research questions:

* **RQ2** — Is upstream aggregate RVI a positive Spearman predictor of
  downstream flood severity?
* **RQ3** — Do quality-verified gauges show a stronger correlation than
  lower-confidence gauges?
* **RQ4** — At which legal buffer threshold (6 m / 10 m / 30 m) does upstream
  RVI best predict flood severity?

**Inputs:**
* `outputs/nairobi/rvi_segments.gpkg` — produced by Notebook 01.
* Live Flood Hub data (or a cached snapshot).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%H:%M:%S')
sys.path.insert(0, '../src')

import geopandas as gpd
import pandas as pd
from rvi.config import get_config
cfg = get_config()
cfg.ensure_dirs()

rvi_path = cfg.outputs_dir / 'nairobi' / 'rvi_segments.gpkg'
rvi_segments = gpd.read_file(rvi_path)
print(f'Loaded {len(rvi_segments)} segments from {rvi_path}')
rvi_segments.filter(like='rvi_composite').describe()

## 1. Fetch Flood Hub gauges + current severity

In [ ]:
from rvi.ingestion.floodhub import FloodHubClient, gauges_to_geodataframe, statuses_to_dataframe, join_gauges_with_status

client = FloodHubClient(config=cfg)
gauges = client.search_gauges_by_area(region_code='KE', include_non_quality_verified=True)
statuses = client.search_latest_flood_status_by_area(region_code='KE', include_non_quality_verified=True)
gauges_geo = gauges_to_geodataframe(gauges)
statuses_df = statuses_to_dataframe(statuses)
gauges_with_status = join_gauges_with_status(gauges_geo, statuses_df)
print(f'Gauges: {len(gauges_with_status)} — quality-verified: {gauges_with_status["quality_verified"].sum()}')
gauges_with_status[['gauge_id','site_name','quality_verified','severity','severity_int']].head()

## 2. RQ4 — multi-width Spearman comparison

We pair each gauge with the upstream 75th-percentile RVI for each of the three
legal buffer widths and report Spearman ρ with a 95% bootstrap CI.

In [ ]:
from rvi.analysis.validation import aggregate_upstream_euclidean, correlate_upstream_rvi_to_severity, stratified_correlation

results = {}
for w in cfg.buffer_widths_m:
    col = f'rvi_composite_{w}m'
    if col not in rvi_segments.columns:
        continue
    seg_view = rvi_segments[['segment_id', col, 'geometry']].rename(columns={col: 'rvi_composite'})
    upstream = aggregate_upstream_euclidean(seg_view, gauges_with_status, config=cfg)
    res = correlate_upstream_rvi_to_severity(upstream, config=cfg)
    results[w] = (upstream, res)
    print(f'{w:>3d} m buffer: ρ = {res.rho:+.3f}  n={res.n}  CI=[{res.ci_low:+.3f}, {res.ci_high:+.3f}]  p={res.pvalue:.3g}')

## 3. RQ3 — quality-verified vs lower-confidence gauges

We pick the buffer width whose all-gauges ρ is highest and stratify by
`quality_verified`.

In [ ]:
if results:
    best_w = max(results, key=lambda w: (results[w][1].rho if results[w][1].rho == results[w][1].rho else -2))
    upstream_best, _ = results[best_w]
    print(f'Best width: {best_w} m')
    strat = stratified_correlation(upstream_best, config=cfg)
    for tier, res in strat.items():
        print(f'  {tier:>22s}: ρ = {res.rho:+.3f}  n={res.n}  CI=[{res.ci_low:+.3f}, {res.ci_high:+.3f}]')

## 4. Output — final scatter (proposal Phase-2 exit criterion)

In [ ]:
from rvi.viz.choropleth import rvi_severity_scatter

if results:
    upstream_best, res_best = results[best_w]
    fig = rvi_severity_scatter(
        upstream_best, result=res_best,
        title=f'Kenya — upstream RVI ({best_w} m buffer) vs Flood Hub severity'
    )
    out_path = cfg.outputs_dir / 'nairobi' / 'rvi_floodhub_correlation_final.png'
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    print('Saved:', out_path)

## 5. Sensitivity (RQ4 follow-on) — weight grid

In [ ]:
from rvi.analysis.encroachment import detect_encroachment
from rvi.analysis.rvi import sensitivity_grid
from rvi.viz.choropleth import sensitivity_heatmap

if 'rvi_composite_30m' in rvi_segments.columns:
    enc30 = detect_encroachment(rvi_segments, gpd.GeoDataFrame(geometry=[]), buffer_width_m=30, config=cfg)
grid = sensitivity_grid(rvi_segments.rename(columns={'rvi_composite_30m':'rvi_composite'}), config=cfg, step=0.1) if False else pd.DataFrame()
if not grid.empty:
    fig = sensitivity_heatmap(grid)
    fig.savefig(cfg.outputs_dir / 'nairobi' / 'rvi_sensitivity_analysis.png', dpi=200, bbox_inches='tight')